# Notebook 01: Acquire, link and screen STATS19 data (2023 to 2025)

**Study:** predicting recorded road casualty severity (KSI) in Great Britain. This notebook implements sections 3.5 and 3.6 of the methodology.

**What it does**
1. Downloads the 9 DfT CSV files and records URL, date, size, row count and SHA-256 checksum for each (manifest).
2. Loads them into SQLite with explicit data-type decisions logged.
3. Runs the conformance, completeness and plausibility checks (Kahn et al., 2016) and logs every result.
4. Links casualty to collision and vehicle, and stops if the join changes the row count.
5. Applies the eligibility rules and writes the exclusion log.
6. Produces the final Table 3.2 counts and compares them with the October 2026 scoping counts.
7. Profiles recording system by year and finds forces whose system changed (needed for the sensitivity check in 3.7.5).

**What it does not do:** no modelling and no evaluation. Run the cells top to bottom. Everything is saved to Google Drive, because Colab runtimes reset.

**Data licence:** Open Government Licence v3.0.

In [1]:
# 1. Setup -----------------------------------------------------------------
import os, sys, hashlib, sqlite3, time, json, subprocess, datetime as dt
from pathlib import Path
import pandas as pd
import requests

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/stats19_severity')
else:
    PROJECT = Path(os.environ.get('PROJECT_DIR', './stats19_severity'))

RAW = PROJECT / 'raw'
DB_PATH = PROJECT / 'db' / 'stats19.sqlite'
OUT = PROJECT / 'outputs' / 'linkage'
ENV = PROJECT / 'environment'
for p in (RAW, DB_PATH.parent, OUT, ENV):
    p.mkdir(parents=True, exist_ok=True)

SEED = 20261107   # reused by every later notebook
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
(ENV / 'pip_freeze_notebook01.txt').write_text(freeze)
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| sqlite', sqlite3.sqlite_version)
print('Project folder:', PROJECT)

Mounted at /content/drive
Python 3.13.18 | pandas 2.2.3 | sqlite 3.45.1
Project folder: /content/drive/MyDrive/stats19_severity


In [2]:
# 2. Configuration -----------------------------------------------------------
YEARS = [2023, 2024, 2025]
DEV_YEARS = [2023, 2024]
TEST_YEAR = 2025
BASE = 'https://data.dft.gov.uk/road-accidents-safety-data/'
KINDS = ['collision', 'vehicle', 'casualty']
FILES = {(k, y): f'dft-road-casualty-statistics-{k}-{y}.csv' for k in KINDS for y in YEARS}

# Columns that must exist. If DfT renames one, the notebook stops and tells you.
REQUIRED = {
    'collision': ['collision_index', 'police_force', 'number_of_vehicles', 'number_of_casualties', 'collision_severity'],
    'vehicle':   ['collision_index', 'vehicle_reference'],
    'casualty':  ['collision_index', 'vehicle_reference', 'casualty_reference', 'casualty_severity'],
}
# Identifier columns stay as text. Everything else is converted to numbers where safe.
ID_COLS = {'collision_index', 'collision_ref_no', 'vehicle_reference', 'casualty_reference'}

# October 2026 scoping counts (Table 3.2), used as a reconciliation check.
SCOPING = {
    2023: dict(casualties=132977, fatal=1624, serious=26030, slight=105323),
    2024: dict(casualties=128272, fatal=1602, serious=26040, slight=100630),
    2025: dict(casualties=127883, fatal=1538, serious=27758, slight=98587),
}

In [3]:
# 3. Download and manifest ----------------------------------------------------
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def download(url, dest, retries=3):
    if dest.exists():
        return 'already present'
    last = None
    for attempt in range(1, retries + 1):
        try:
            tmp = dest.with_suffix(dest.suffix + '.part')
            with requests.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
            tmp.rename(dest)
            return 'downloaded'
        except Exception as e:
            last = e
            time.sleep(5 * attempt)
    raise RuntimeError(f'Could not download {url}: {last}')

records = []
for (kind, year), fname in FILES.items():
    dest = RAW / fname
    status = download(BASE + fname, dest)
    n_rows = len(pd.read_csv(dest, usecols=[0], dtype=str))
    records.append(dict(
        kind=kind, year=year, file=fname, url=BASE + fname, status=status,
        recorded_utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'),
        bytes=dest.stat().st_size, rows=n_rows, sha256=sha256_of(dest)))

manifest = pd.DataFrame(records)
hist_path = OUT / 'manifest_history.csv'
if hist_path.exists():
    hist = pd.read_csv(hist_path, dtype=str)
    prev = hist.sort_values('recorded_utc').groupby('file').tail(1).set_index('file')['sha256']
    for _, r in manifest.iterrows():
        if r['file'] in prev.index and prev[r['file']] != r['sha256']:
            print('WARNING: checksum changed since last run for', r['file'])
    pd.concat([hist, manifest.astype(str)]).to_csv(hist_path, index=False)
else:
    manifest.to_csv(hist_path, index=False)
manifest.to_csv(OUT / 'manifest_current.csv', index=False)
manifest[['kind', 'year', 'status', 'bytes', 'rows', 'sha256']]

,kind,year,status,bytes,rows,sha256
0,collision,2023,downloaded,19790001,104258,0c01d4194b4e4c5e1b242a43c633bc30e22cf16e8becca...
1,collision,2024,downloaded,19208488,100927,32a78f4662e309160f37b1ea2b82017e85e07b62a022d6...
2,collision,2025,downloaded,19349153,101525,e699c19bdb82e47e449b6f3381ac87631865f6a31cef11...
3,vehicle,2023,downloaded,20914478,189815,9beafb1831cc611bcc0337eddaeb8b1cbdf03440df3be1...
4,vehicle,2024,downloaded,20296911,183514,1a60f65f07a44454bc03c63b35d6741f41c9dfbb05a267...
5,vehicle,2025,downloaded,20688988,183948,dbeadb9745a8821ab8132f0dde59c246159e23059ab5db...
6,casualty,2023,downloaded,11012867,132977,83e509384d877535bcd566772d029a1408568114fe51a5...
7,casualty,2024,downloaded,10439101,128272,838aad605a6ac638973f09248ef3a730dbd1de30ac8fc7...
8,casualty,2025,downloaded,10318523,127883,b343039991f63f526bbcac0a5e6102e3a6e4c0b65d077e...


In [4]:
# 4. Load into SQLite with logged data-type decisions ---------------------------
def load_table(kind):
    frames = []
    for year in YEARS:
        df = pd.read_csv(RAW / FILES[(kind, year)], dtype=str, low_memory=False)
        df.columns = [c.strip().lower() for c in df.columns]
        df['source_year'] = year
        frames.append(df)
    # which columns are missing in some years?
    all_cols = set().union(*[set(f.columns) for f in frames])
    coverage = [dict(table=kind, column=c, years_present=','.join(str(y) for y, f in zip(YEARS, frames) if c in f.columns))
                for c in sorted(all_cols) if not all(c in f.columns for f in frames)]
    df = pd.concat(frames, ignore_index=True)

    missing_req = [c for c in REQUIRED[kind] if c not in df.columns]
    if missing_req:
        similar = {m: [c for c in df.columns if m.split('_')[0] in c][:5] for m in missing_req}
        raise KeyError(f'{kind}: required column(s) {missing_req} not found. Similar names: {similar}')
    if 'collision_year' not in df.columns:
        df['collision_year'] = df['source_year']

    decisions = []
    for c in df.columns:
        if c == 'source_year':
            decisions.append(dict(table=kind, column=c, decision='integer (file year)', non_numeric_values=0))
            continue
        if c in ID_COLS:
            df[c] = df[c].astype('string').str.strip()
            decisions.append(dict(table=kind, column=c, decision='text (identifier)', non_numeric_values=0))
            continue
        s = df[c]
        num = pd.to_numeric(s, errors='coerce')
        n_present, n_numeric = s.notna().sum(), num.notna().sum()
        if n_present > 0 and n_numeric >= 0.999 * n_present:
            df[c] = num
            decisions.append(dict(table=kind, column=c, decision='numeric', non_numeric_values=int(n_present - n_numeric)))
        else:
            df[c] = s.astype('string').str.strip()
            decisions.append(dict(table=kind, column=c, decision='text', non_numeric_values=int(n_present - n_numeric)))
    return df, pd.DataFrame(decisions), pd.DataFrame(coverage)

if DB_PATH.exists():
    DB_PATH.unlink()
con = sqlite3.connect(DB_PATH)
all_decisions, all_coverage = [], []
for kind in KINDS:
    df, dec, cov = load_table(kind)
    df.to_sql(kind, con, index=False, chunksize=50000)
    all_decisions.append(dec); all_coverage.append(cov)
    print(f'{kind}: {len(df):,} rows, {df.shape[1]} columns')
pd.concat(all_decisions).to_csv(OUT / 'dtype_decisions.csv', index=False)
pd.concat(all_coverage).to_csv(OUT / 'column_coverage_by_year.csv', index=False)

con.executescript("""
CREATE INDEX ix_col ON collision(collision_index);
CREATE INDEX ix_veh ON vehicle(collision_index, vehicle_reference);
CREATE INDEX ix_cas ON casualty(collision_index, casualty_reference);
""")
cov = pd.concat(all_coverage)
print('\nColumns not present in every year (check the data guide if any matter):')
cov

collision: 306,710 rows, 45 columns
vehicle: 557,277 rows, 33 columns
casualty: 389,132 rows, 25 columns

Columns not present in every year (check the data guide if any matter):


""


In [5]:
# 5. Conformance, completeness and plausibility checks ---------------------------
checks = []
def run_check(dimension, name, sql, note=''):
    res = pd.read_sql(sql, con)
    for _, r in res.iterrows():
        checks.append(dict(dimension=dimension, check=name, year=r['source_year'], n_flagged=int(r['n']), note=note))

for t in KINDS:
    run_check('conformance', f'{t}: null or blank collision_index',
              f"SELECT source_year, COUNT(*) AS n FROM {t} WHERE collision_index IS NULL OR TRIM(collision_index)='' GROUP BY 1")

run_check('conformance', 'collision: duplicate collision_index',
          "SELECT 'all' AS source_year, COUNT(*) AS n FROM (SELECT collision_index FROM collision GROUP BY 1 HAVING COUNT(*)>1)")
run_check('conformance', 'vehicle: duplicate (collision_index, vehicle_reference)',
          "SELECT 'all' AS source_year, COUNT(*) AS n FROM (SELECT 1 FROM vehicle GROUP BY collision_index, vehicle_reference HAVING COUNT(*)>1)")
run_check('conformance', 'casualty: duplicate (collision_index, casualty_reference)',
          "SELECT 'all' AS source_year, COUNT(*) AS n FROM (SELECT 1 FROM casualty GROUP BY collision_index, casualty_reference HAVING COUNT(*)>1)")
run_check('conformance', 'casualty: no matching collision',
          "SELECT c.source_year, COUNT(*) AS n FROM casualty c LEFT JOIN collision x ON c.collision_index=x.collision_index WHERE x.collision_index IS NULL GROUP BY 1")
run_check('conformance', 'casualty: no matching vehicle (kept, vehicle fields set unavailable)',
          "SELECT c.source_year, COUNT(*) AS n FROM casualty c LEFT JOIN vehicle v ON c.collision_index=v.collision_index AND c.vehicle_reference=v.vehicle_reference WHERE v.collision_index IS NULL GROUP BY 1",
          'Includes pedestrians where no vehicle reference applies; inspect before deciding.')
run_check('conformance', 'casualty: severity code not 1, 2 or 3',
          "SELECT source_year, COUNT(*) AS n FROM casualty WHERE casualty_severity IS NULL OR CAST(casualty_severity AS INTEGER) NOT IN (1,2,3) GROUP BY 1")
run_check('conformance', 'collision: collision_year differs from file year',
          "SELECT source_year, COUNT(*) AS n FROM collision WHERE CAST(collision_year AS INTEGER) <> source_year GROUP BY 1")
run_check('plausibility', 'collision: number_of_casualties differs from casualty rows',
          """SELECT x.source_year, COUNT(*) AS n FROM collision x
             JOIN (SELECT collision_index, COUNT(*) AS k FROM casualty GROUP BY 1) y ON x.collision_index=y.collision_index
             WHERE CAST(x.number_of_casualties AS INTEGER) <> y.k GROUP BY 1""")
run_check('plausibility', 'collision: number_of_vehicles differs from vehicle rows',
          """SELECT x.source_year, COUNT(*) AS n FROM collision x
             JOIN (SELECT collision_index, COUNT(*) AS k FROM vehicle GROUP BY 1) y ON x.collision_index=y.collision_index
             WHERE CAST(x.number_of_vehicles AS INTEGER) <> y.k GROUP BY 1""")
cas_cols = pd.read_sql('PRAGMA table_info(casualty)', con)['name'].tolist()
if 'age_of_casualty' in cas_cols:
    run_check('plausibility', 'casualty: age above 110 (unknown age is coded -1 and not flagged)',
              "SELECT source_year, COUNT(*) AS n FROM casualty WHERE CAST(age_of_casualty AS INTEGER) > 110 GROUP BY 1")

checks_df = pd.DataFrame(checks)
checks_df.to_csv(OUT / 'linkage_checks.csv', index=False)
checks_df.pivot_table(index=['dimension', 'check'], columns='year', values='n_flagged', aggfunc='sum', fill_value=0)

year                                                            all
dimension   check                                                  
conformance casualty: duplicate (collision_index, casualty_...    0
            collision: duplicate collision_index                  0
            vehicle: duplicate (collision_index, vehicle_re...    0

In [6]:
# 6. Linkage and eligibility ---------------------------------------------------
col_cols = pd.read_sql('PRAGMA table_info(collision)', con)['name'].tolist()
veh_cols = pd.read_sql('PRAGMA table_info(vehicle)', con)['name'].tolist()
cas_cols = pd.read_sql('PRAGMA table_info(casualty)', con)['name'].tolist()
col_extra = [c for c in col_cols if c not in cas_cols]
veh_extra = [c for c in veh_cols if c not in cas_cols and c not in col_cols]
col_sel = ', '.join(f'x."{c}"' for c in col_extra)
veh_sel = ', '.join(f'v."{c}"' for c in veh_extra)

con.executescript(f"""
DROP TABLE IF EXISTS casualty_linked;
CREATE TABLE casualty_linked AS
SELECT c.*,
       {col_sel},
       {veh_sel},
       CASE WHEN x.collision_index IS NULL THEN 0 ELSE 1 END AS collision_match,
       CASE WHEN v.collision_index IS NULL THEN 0 ELSE 1 END AS vehicle_match
FROM casualty c
LEFT JOIN collision x ON c.collision_index = x.collision_index
LEFT JOIN vehicle v ON c.collision_index = v.collision_index AND c.vehicle_reference = v.vehicle_reference;
""")

n_cas = con.execute('SELECT COUNT(*) FROM casualty').fetchone()[0]
n_linked = con.execute('SELECT COUNT(*) FROM casualty_linked').fetchone()[0]
print(f'casualty rows: {n_cas:,} | after linkage: {n_linked:,}')
if n_cas != n_linked:
    dup = pd.read_sql("""SELECT collision_index, vehicle_reference, COUNT(*) AS n FROM vehicle
                         GROUP BY 1,2 HAVING COUNT(*)>1 LIMIT 10""", con)
    raise RuntimeError('The join changed the number of casualty rows (duplicate vehicle keys?). Examples:\n' + dup.to_string())

con.executescript("""
DROP TABLE IF EXISTS casualty_eligibility;
CREATE TABLE casualty_eligibility AS
SELECT *,
  CASE
    WHEN collision_index IS NULL OR TRIM(collision_index)='' OR casualty_reference IS NULL OR TRIM(casualty_reference)='' THEN '1 invalid casualty key'
    WHEN key_count > 1 THEN '2 duplicate casualty key'
    WHEN casualty_severity IS NULL OR CAST(casualty_severity AS INTEGER) NOT IN (1,2,3) THEN '3 unrecognised severity code'
    WHEN collision_match = 0 THEN '4 no collision match'
    ELSE 'eligible'
  END AS eligibility
FROM (SELECT l.*, COUNT(*) OVER (PARTITION BY collision_index, casualty_reference) AS key_count FROM casualty_linked l);
DROP TABLE IF EXISTS casualty_eligible;
CREATE TABLE casualty_eligible AS SELECT * FROM casualty_eligibility WHERE eligibility = 'eligible';
CREATE INDEX ix_elig_year ON casualty_eligible(source_year);
""")

excl = pd.read_sql("SELECT source_year AS year, eligibility, COUNT(*) AS n FROM casualty_eligibility GROUP BY 1,2", con)
excl_table = excl.pivot_table(index='eligibility', columns='year', values='n', aggfunc='sum', fill_value=0)
excl_table.to_csv(OUT / 'exclusion_log.csv')
vm = pd.read_sql("SELECT source_year AS year, SUM(1-vehicle_match) AS casualties_without_vehicle_match, COUNT(*) AS eligible FROM casualty_eligible GROUP BY 1", con)
vm.to_csv(OUT / 'unmatched_vehicle_by_year.csv', index=False)
print('\nEligibility by year'); display(excl_table)
print('\nEligible casualties without a vehicle match (kept, vehicle fields unavailable)'); display(vm)

casualty rows: 389,132 | after linkage: 389,132

Eligibility by year


year,2023,2024,2025
eligibility,,,
eligible,132977,128272,127883



Eligible casualties without a vehicle match (kept, vehicle fields unavailable)


,year,casualties_without_vehicle_match,eligible
0,2023,0,132977
1,2024,0,128272
2,2025,0,127883


In [7]:
# 7. Table 3.2 counts and reconciliation with scoping ----------------------------
t32 = pd.read_sql("""
SELECT source_year AS year, COUNT(*) AS casualties,
       SUM(CAST(casualty_severity AS INTEGER)=1) AS fatal,
       SUM(CAST(casualty_severity AS INTEGER)=2) AS serious,
       SUM(CAST(casualty_severity AS INTEGER)=3) AS slight
FROM casualty_eligible GROUP BY 1 ORDER BY 1""", con).set_index('year')
dev = t32.loc[DEV_YEARS].sum().rename('2023-2024 development total')
t32 = pd.concat([t32, dev.to_frame().T])
t32['recorded_KSI_pct'] = ((t32['fatal'] + t32['serious']) / t32['casualties'] * 100).round(1)
t32.to_csv(OUT / 'table_3_2_final_counts.csv')
display(t32)

rows = []
for y, exp in SCOPING.items():
    for k, v in exp.items():
        got = int(t32.loc[y, k]) if y in t32.index else None
        rows.append(dict(year=y, measure=k, scoping=v, final=got, difference=None if got is None else got - v))
recon = pd.DataFrame(rows)
recon.to_csv(OUT / 'reconciliation_with_scoping.csv', index=False)
print('Reconciliation with the October 2026 scoping counts (non-zero differences need explaining in 3.5):')
display(recon[recon['difference'] != 0] if (recon['difference'] != 0).any() else pd.DataFrame({'result': ['All counts match the scoping figures']}))

,casualties,fatal,serious,slight,recorded_KSI_pct
2023,132977,1624,26030,105323,20.8
2024,128272,1602,26040,100630,21.5
2025,127883,1538,27758,98587,22.9
2023-2024 development total,261249,3226,52070,205953,21.2


Reconciliation with the October 2026 scoping counts (non-zero differences need explaining in 3.5):


,result
0,All counts match the scoping figures


In [8]:
# 8. Recording system by year, KSI rate by system, and forces that changed system ---
elig = pd.read_sql("SELECT * FROM casualty_eligible", con)
FLAG = 'casualty_injury_based' if 'casualty_injury_based' in elig.columns else None
if FLAG is None:
    cands = [c for c in elig.columns if 'injury_based' in c]
    FLAG = cands[0] if cands else None
print('Injury-based flag used:', FLAG, '| CONFIRM the coding (1 = injury-based?) against the 2025 data guide.')

HI, LO = 0.90, 0.10     # a force-year counts as injury-based at or above HI and non-injury-based at or below LO
if FLAG:
    elig['ksi'] = (pd.to_numeric(elig['casualty_severity']) <= 2).astype(int)
    flag_num = pd.to_numeric(elig[FLAG], errors='coerce')
    print('Flag values (all years):', flag_num.value_counts(dropna=False).to_dict())
    valid = elig[flag_num.isin([0, 1])].copy()
    valid[FLAG] = flag_num[valid.index]

    # (a) Share of casualties by recording system, with KSI rate (descriptive only)
    by_year = (valid.groupby(['source_year', FLAG]).agg(casualties=('ksi', 'size'), ksi_rate=('ksi', 'mean')).reset_index())
    by_year['share_of_year'] = by_year['casualties'] / by_year.groupby('source_year')['casualties'].transform('sum')
    by_year.round(4).to_csv(OUT / 'recording_system_by_year.csv', index=False)
    print('\nCasualties, share and recorded KSI rate by recording system and year:')
    display(by_year.round(3))

    # (b) Force-year share of casualties recorded injury-based, then a stable/changed grouping
    share = valid.groupby(['police_force', 'source_year'])[FLAG].mean().unstack()
    def label(x):
        if pd.isna(x): return 'no data'
        return 'injury-based' if x >= HI else ('non-injury-based' if x <= LO else 'mixed')
    classes = share.apply(lambda col: col.map(label))
    force = share.add_prefix('share_injury_based_').join(classes.add_prefix('class_'))
    def group(row):
        cl = [row[f'class_{y}'] for y in YEARS]
        if 'mixed' in cl or 'no data' in cl or len(set(cl)) > 1:
            return 'changed or mixed'
        return 'stable ' + cl[0]
    force['system_group'] = force.apply(group, axis=1)
    force.reset_index().to_csv(OUT / 'forces_recording_system_2023_2025.csv', index=False)

    valid = valid.merge(force[['system_group']], left_on='police_force', right_index=True, how='left')
    grp = (valid.groupby(['system_group', 'source_year']).agg(casualties=('ksi', 'size'), ksi_rate=('ksi', 'mean'))
           .reset_index().round(4))
    grp.to_csv(OUT / 'recording_system_groups_by_year.csv', index=False)
    print('\nForces by recording-system group:', force['system_group'].value_counts().to_dict())
    print('\nCasualties and KSI rate by force group and year (use the stable groups for the sensitivity check in 3.7.5):')
    display(grp)
    print('\nPolice force codes: map them to names with the data guide.')

Injury-based flag used: casualty_injury_based | CONFIRM the coding (1 = injury-based?) against the 2025 data guide.
Flag values (all years): {1: 286129, 0: 103003}

Casualties, share and recorded KSI rate by recording system and year:


,source_year,casualty_injury_based,casualties,ksi_rate,share_of_year
0,2023,0,58820,0.171,0.442
1,2023,1,74157,0.237,0.558
2,2024,0,26789,0.198,0.209
3,2024,1,101483,0.220,0.791
4,2025,0,17394,0.187,0.136
5,2025,1,110489,0.236,0.864



Forces by recording-system group: {'stable injury-based': 26, 'changed or mixed': 13, 'stable non-injury-based': 5}

Casualties and KSI rate by force group and year (use the stable groups for the sensitivity check in 3.7.5):


,system_group,source_year,casualties,ksi_rate
0,changed or mixed,2023,51297,0.1669
1,changed or mixed,2024,48630,0.1778
2,changed or mixed,2025,49066,0.2048
3,stable injury-based,2023,73119,0.2355
4,stable injury-based,2024,71737,0.2415
5,stable injury-based,2025,70735,0.2466
6,stable non-injury-based,2023,8561,0.2191
7,stable non-injury-based,2024,7905,0.2116
8,stable non-injury-based,2025,8082,0.2236



Police force codes: map them to names with the data guide.


In [9]:
# 9. Missing and unknown-code profile (feeds the field dictionary) -----------------
num_cols = [c for c in elig.columns if pd.api.types.is_numeric_dtype(elig[c]) and c not in ('source_year',)]
rows = []
for y, g in elig.groupby('source_year'):
    for c in num_cols:
        s = g[c]
        rows.append(dict(year=y, column=c, n=len(g), null=int(s.isna().sum()), coded_minus_one=int((s == -1).sum())))
miss = pd.DataFrame(rows)
miss['pct_unknown'] = ((miss['null'] + miss['coded_minus_one']) / miss['n'] * 100).round(2)
miss.to_csv(OUT / 'missing_profile_by_year.csv', index=False)
# Fields that are almost empty in one year but populated in another (retired or newly introduced fields)
piv = miss.pivot(index='column', columns='year', values='pct_unknown')
drift = piv[(piv.max(axis=1) >= 95) & (piv.min(axis=1) <= 50)].round(1)
drift.to_csv(OUT / 'schema_drift_fields.csv')
print('Fields near-empty in at least one year but populated in another (do not use without harmonising):')
display(drift)
print('Fields with the highest share of null or -1 (DfT code for missing or out of range):')
display(miss.sort_values('pct_unknown', ascending=False).head(15))

Fields near-empty in at least one year but populated in another (do not use without harmonising):


year,2023,2024,2025
column,,,
age_of_vehicle,21.5,27.2,100.0
casualty_distance_banding,100.0,13.5,11.5
driver_distance_banding,100.0,12.4,13.3


Fields with the highest share of null or -1 (DfT code for missing or out of range):


,year,column,n,null,coded_minus_one,pct_unknown
17,2023,casualty_distance_banding,132977,0,132977,100.00
28,2023,local_authority_district,132977,0,132977,100.00
78,2023,driver_distance_banding,132977,0,132977,100.00
111,2024,local_authority_district,128272,0,128272,100.00
242,2025,age_of_vehicle,127883,0,127883,100.00
194,2025,local_authority_district,127883,0,127883,100.00
222,2025,vehicle_manoeuvre_historic,127883,0,112163,87.71
235,2025,journey_purpose_of_driver_historic,127883,0,111772,87.40
199,2025,junction_detail_historic,127883,0,110513,86.42
210,2025,special_conditions_at_site,127883,0,110502,86.41


In [10]:
# 10. Summary and export -------------------------------------------------------
failing = checks_df[(checks_df['n_flagged'] > 0)].groupby('check')['n_flagged'].sum().to_dict()
summary = dict(
    run_utc=dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'),
    seed=SEED,
    casualties_raw=int(n_cas),
    casualties_eligible=int(con.execute('SELECT COUNT(*) FROM casualty_eligible').fetchone()[0]),
    checks_with_flags=failing,
    files=manifest[['file', 'sha256']].to_dict('records'),
)
(OUT / 'linkage_summary.json').write_text(json.dumps(summary, indent=2, default=str))
con.commit(); con.close()
print(json.dumps({k: v for k, v in summary.items() if k != 'files'}, indent=2, default=str))
print('\nSaved to', OUT)
for p in sorted(OUT.glob('*')):
    print(' -', p.name)
print('\nNEXT: notebook 02 (preprocessing and saved folds). Do not load 2025 labels in any modelling step yet.')

{
  "run_utc": "2026-11-07T11:38:20+00:00",
  "seed": 20261107,
  "casualties_raw": 389132,
  "casualties_eligible": 389132,
  "checks_with_flags": {}
}

Saved to /content/drive/MyDrive/stats19_severity/outputs/linkage
 - column_coverage_by_year.csv
 - dtype_decisions.csv
 - exclusion_log.csv
 - forces_recording_system_2023_2025.csv
 - linkage_checks.csv
 - linkage_summary.json
 - manifest_current.csv
 - manifest_history.csv
 - missing_profile_by_year.csv
 - reconciliation_with_scoping.csv
 - recording_system_by_year.csv
 - recording_system_groups_by_year.csv
 - schema_drift_fields.csv
 - table_3_2_final_counts.csv
 - unmatched_vehicle_by_year.csv

NEXT: notebook 02 (preprocessing and saved folds). Do not load 2025 labels in any modelling step yet.
